# Team Ultron Medical Abstract Classification Benchmark

Canonical final results: BiomedBERT validation Macro F1 **0.8672**; locked test Macro F1 **0.8603**; hierarchical validation Macro F1 **0.8599**; ECE **0.1041**; throughput **35.62 abstracts/s** on a Colab T4.

Stored execution outputs were removed for the repository snapshot. See `results/` for the authoritative machine-readable records.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Chosen before model comparison because the classes are imbalanced.
PRIMARY_METRIC = "Macro F1"

def evaluate_model(name, y_true, y_pred):
    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Weighted F1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        )
    }
    return results

print("Primary metric:", PRIMARY_METRIC)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    max_features=30000,
    ngram_range=(1, 2)
)

X_train = tfidf.fit_transform(train_df["medical_abstract"])
X_val = tfidf.transform(val_df["medical_abstract"])

y_train = train_df["condition_label"]
y_val = val_df["condition_label"]

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_val.shape)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_val)

nb_results = evaluate_model(
    "Multinomial Naive Bayes",
    y_val,
    nb_predictions
)

print("Validation metrics:")
for metric, value in nb_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.6f}")

print("\nClassification report:")
print(classification_report(y_val, nb_predictions, digits=4))

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight=None,
    random_state=42
)

lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_val)

lr_results = evaluate_model(
    "Logistic Regression",
    y_val,
    lr_predictions
)

print("Validation metrics:")
for metric, value in lr_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.6f}")

print("\nClassification report:")
print(classification_report(
    y_val,
    lr_predictions,
    digits=4
))

In [ ]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC()

svm_model.fit(X_train, y_train)

svm_predictions = svm_model.predict(X_val)

svm_results = evaluate_model(
    "Linear SVM",
    y_val,
    svm_predictions
)

print("Validation metrics:")
for metric, value in svm_results.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.6f}")

print("\nClassification report:")
print(classification_report(
    y_val,
    svm_predictions,
    digits=4
))

In [ ]:
baseline_results = pd.DataFrame([
    nb_results,
    lr_results,
    svm_results
])

baseline_results = baseline_results.sort_values(
    by="Macro F1",
    ascending=False
)

display(baseline_results)

In [ ]:
baseline_results.to_csv(
    "baseline_validation_results.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

labels = [
    "Neoplasms",
    "Digestive",
    "Nervous",
    "Cardiovascular",
    "General"
]

ConfusionMatrixDisplay.from_predictions(
    y_val,
    svm_predictions,
    display_labels=labels,
    xticks_rotation=45,
    cmap="Blues"
)

plt.title("Linear SVM - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
svm_confusion_matrix = pd.DataFrame(confusion_matrix(y_val, svm_predictions, labels=[1, 2, 3, 4, 5]), index=labels, columns=labels)
print("Rows = actual labels; columns = predicted labels")
print(svm_confusion_matrix.to_string())

In [ ]:
# Validation-only tuning: the test set is deliberately not referenced.
tuning_records = []

for c_value in [0.1, 0.5, 1.0, 2.0, 5.0]:
    model = LinearSVC(C=c_value, random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    result = evaluate_model("Linear SVM", y_val, predictions)
    result.update({"C": c_value, "Class Weight": "None"})
    tuning_records.append(result)

for c_value in [0.1, 0.5, 1.0, 2.0, 5.0]:
    for class_weight in [None, "balanced"]:
        model = LogisticRegression(
            C=c_value,
            class_weight=class_weight,
            max_iter=1000,
            random_state=42
        )
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)
        result = evaluate_model("Logistic Regression", y_val, predictions)
        result.update({
            "C": c_value,
            "Class Weight": "None" if class_weight is None else class_weight
        })
        tuning_records.append(result)

tuning_results = pd.DataFrame(tuning_records)[[
    "Model", "C", "Class Weight", "Accuracy", "Macro Precision",
    "Macro Recall", "Macro F1", "Weighted F1"
]].sort_values("Macro F1", ascending=False).reset_index(drop=True)

tuning_results.to_csv("tuning_validation_results.csv", index=False)

print("All validation tuning experiments (ranked by Macro F1):")
print(tuning_results.to_string(index=False))
print()
print("Best validation configuration:")
print(tuning_results.iloc[0].to_string())

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Validation-only tuning.
# The test set is NOT used here.

tuning_records = []

# -----------------------------
# 1. Logistic Regression tuning
# -----------------------------
for c_value in [0.1, 0.5, 1.0, 2.0, 5.0]:
    for class_weight in [None, "balanced"]:

        model = LogisticRegression(
            C=c_value,
            class_weight=class_weight,
            max_iter=1000,
            random_state=42
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        result = evaluate_model(
            "Logistic Regression",
            y_val,
            predictions
        )

        result.update({
            "C": c_value,
            "Class Weight": "None" if class_weight is None else "balanced"
        })

        tuning_records.append(result)


# -----------------------------
# 2. Linear SVM tuning
# -----------------------------
for c_value in [0.1, 0.5, 1.0, 2.0, 5.0]:

    model = LinearSVC(
        C=c_value,
        random_state=42
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_val)

    result = evaluate_model(
        "Linear SVM",
        y_val,
        predictions
    )

    result.update({
        "C": c_value,
        "Class Weight": "None"
    })

    tuning_records.append(result)


# -----------------------------
# 3. Create ranked results table
# -----------------------------
tuning_results = pd.DataFrame(tuning_records)

tuning_results = tuning_results[
    [
        "Model",
        "C",
        "Class Weight",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ]
]

tuning_results = tuning_results.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)


# -----------------------------
# 4. Save experiment results
# -----------------------------
tuning_results.to_csv(
    "tuning_validation_results.csv",
    index=False
)


# -----------------------------
# 5. Show results
# -----------------------------
print("All validation tuning experiments ranked by Macro F1:\n")
display(tuning_results)

print("\nBest validation configuration:")
display(tuning_results.iloc[[0]])

print("\nSaved as: tuning_validation_results.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use training data only for model-readiness decisions
word_lengths = (
    train_df["medical_abstract"]
    .astype(str)
    .str.split()
    .str.len()
)

print("TRAINING ABSTRACT LENGTH ANALYSIS")
print("-" * 45)
print(f"Number of abstracts: {len(word_lengths)}")
print(f"Minimum words: {word_lengths.min()}")
print(f"Mean words: {word_lengths.mean():.2f}")
print(f"Median words: {word_lengths.median():.2f}")
print(f"75th percentile: {np.percentile(word_lengths, 75):.0f}")
print(f"90th percentile: {np.percentile(word_lengths, 90):.0f}")
print(f"95th percentile: {np.percentile(word_lengths, 95):.0f}")
print(f"97.5th percentile: {np.percentile(word_lengths, 97.5):.0f}")
print(f"99th percentile: {np.percentile(word_lengths, 99):.0f}")
print(f"Maximum words: {word_lengths.max()}")

print("\nPotential truncation by word length")
print("-" * 45)

for limit in [128, 256, 384, 512]:
    count = (word_lengths > limit).sum()
    pct = count / len(word_lengths) * 100
    print(f"> {limit:3d} words: {count:4d} abstracts ({pct:.2f}%)")

plt.figure(figsize=(9, 5))
plt.hist(word_lengths, bins=50)
plt.xlabel("Abstract length (words)")
plt.ylabel("Number of abstracts")
plt.title("Training Abstract Length Distribution")
plt.tight_layout()
plt.show()

In [ ]:
length_summary = pd.DataFrame({
    "Statistic": [
        "count",
        "min",
        "mean",
        "median",
        "75th_percentile",
        "90th_percentile",
        "95th_percentile",
        "97.5th_percentile",
        "99th_percentile",
        "max"
    ],
    "Value": [
        len(word_lengths),
        word_lengths.min(),
        word_lengths.mean(),
        word_lengths.median(),
        np.percentile(word_lengths, 75),
        np.percentile(word_lengths, 90),
        np.percentile(word_lengths, 95),
        np.percentile(word_lengths, 97.5),
        np.percentile(word_lengths, 99),
        word_lengths.max()
    ]
})

length_summary.to_csv(
    "abstract_length_analysis.csv",
    index=False
)

display(length_summary)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd
import time

X_train_text = train_df["medical_abstract"].astype(str)
y_train = train_df["condition_label"]

X_val_text = val_df["medical_abstract"].astype(str)
y_val = val_df["condition_label"]

feature_experiments = {
    "word_unigram": TfidfVectorizer(
        lowercase=True,
        max_features=30000,
        ngram_range=(1, 1)
    ),

    "word_unigram_bigram": TfidfVectorizer(
        lowercase=True,
        max_features=30000,
        ngram_range=(1, 2)
    ),

    "word_plus_char": FeatureUnion([
        (
            "word",
            TfidfVectorizer(
                lowercase=True,
                max_features=30000,
                ngram_range=(1, 2)
            )
        ),
        (
            "char",
            TfidfVectorizer(
                analyzer="char_wb",
                lowercase=True,
                max_features=20000,
                ngram_range=(3, 5)
            )
        )
    ])
}

models = {
    "Logistic Regression": LogisticRegression(
        C=2,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Linear SVM": LinearSVC(
        C=0.5
    )
}

results = []

for feature_name, vectorizer in feature_experiments.items():

    print(f"\n{'='*60}")
    print(f"FEATURE SET: {feature_name}")
    print("="*60)

    X_train_features = vectorizer.fit_transform(X_train_text)
    X_val_features = vectorizer.transform(X_val_text)

    print("Train shape:", X_train_features.shape)
    print("Validation shape:", X_val_features.shape)

    for model_name, model in models.items():

        start = time.time()

        model.fit(X_train_features, y_train)
        predictions = model.predict(X_val_features)

        elapsed = time.time() - start

        row = {
            "Feature_Set": feature_name,
            "Model": model_name,
            "Accuracy": accuracy_score(y_val, predictions),
            "Macro_Precision": precision_score(
                y_val, predictions, average="macro"
            ),
            "Macro_Recall": recall_score(
                y_val, predictions, average="macro"
            ),
            "Macro_F1": f1_score(
                y_val, predictions, average="macro"
            ),
            "Weighted_F1": f1_score(
                y_val, predictions, average="weighted"
            ),
            "Training_Time_Seconds": elapsed
        }

        results.append(row)

        print(
            f"{model_name}: "
            f"Macro F1 = {row['Macro_F1']:.4f}, "
            f"Accuracy = {row['Accuracy']:.4f}"
        )

feature_results_df = pd.DataFrame(results)

feature_results_df = feature_results_df.sort_values(
    by="Macro_F1",
    ascending=False
).reset_index(drop=True)

display(feature_results_df)

feature_results_df.to_csv(
    "feature_experiment_results.csv",
    index=False
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Winning classical representation
best_vectorizer = TfidfVectorizer(
    lowercase=True,
    max_features=30000,
    ngram_range=(1, 1)
)

X_train_best = best_vectorizer.fit_transform(
    train_df["medical_abstract"].astype(str)
)

X_val_best = best_vectorizer.transform(
    val_df["medical_abstract"].astype(str)
)

# Winning classical model
best_classical_model = LogisticRegression(
    C=2,
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

best_classical_model.fit(
    X_train_best,
    train_df["condition_label"]
)

val_predictions = best_classical_model.predict(X_val_best)

print("BEST CLASSICAL MODEL")
print("TF-IDF Word Unigrams + Logistic Regression")
print()

print(
    classification_report(
        val_df["condition_label"],
        val_predictions,
        digits=4
    )
)

cm = confusion_matrix(
    val_df["condition_label"],
    val_predictions
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Neoplasms",
        "Digestive",
        "Nervous",
        "Cardiovascular",
        "General"
    ]
)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=30)
plt.title("Best Classical Model - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# PHASE 2 — TEXTCNN
# =========================

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import re
from collections import Counter

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
def simple_tokenize(text):
    text = str(text).lower()
    return re.findall(r"\b\w+\b", text)

train_tokens = [
    simple_tokenize(text)
    for text in train_df["medical_abstract"]
]

word_counter = Counter()
for tokens in train_tokens:
    word_counter.update(tokens)

MIN_FREQ = 2

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, freq in word_counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

In [ ]:
MAX_LEN = 384

class MedicalAbstractDataset(Dataset):
    def __init__(self, dataframe, vocab, max_len):
        self.texts = dataframe["medical_abstract"].astype(str).tolist()
        self.labels = dataframe["condition_label"].astype(int).values - 1
        self.vocab = vocab
        self.max_len = max_len

    def encode(self, text):
        tokens = simple_tokenize(text)

        ids = [
            self.vocab.get(token, self.vocab["<UNK>"])
            for token in tokens
        ]

        ids = ids[:self.max_len]

        if len(ids) < self.max_len:
            ids += [self.vocab["<PAD>"]] * (self.max_len - len(ids))

        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = self.encode(self.texts[idx])
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

In [ ]:
BATCH_SIZE = 64

train_dataset = MedicalAbstractDataset(
    train_df,
    vocab,
    MAX_LEN
)

val_dataset = MedicalAbstractDataset(
    val_df,
    vocab,
    MAX_LEN
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        num_classes=5,
        num_filters=100,
        filter_sizes=(3, 4, 5),
        dropout=0.5,
        padding_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=fs
            )
            for fs in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(filter_sizes),
            num_classes
        )

    def forward(self, x):
        # x: [batch_size, seq_len]

        x = self.embedding(x)
        # [batch_size, seq_len, embedding_dim]

        x = x.permute(0, 2, 1)
        # [batch_size, embedding_dim, seq_len]

        conv_outputs = []

        for conv in self.convs:
            c = F.relu(conv(x))
            # [batch_size, num_filters, new_seq_len]

            c = F.max_pool1d(
                c,
                kernel_size=c.shape[2]
            ).squeeze(2)

            conv_outputs.append(c)

        x = torch.cat(conv_outputs, dim=1)

        x = self.dropout(x)

        return self.fc(x)

In [ ]:
torch.manual_seed(42)

model = TextCNN(
    vocab_size=len(vocab),
    embedding_dim=128,
    num_classes=5,
    num_filters=100,
    filter_sizes=(3, 4, 5),
    dropout=0.5
).to(device)

print(model)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

EPOCHS = 10

In [ ]:
from copy import deepcopy

best_val_f1 = -1
best_model_state = None

history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": []
}

for epoch in range(EPOCHS):

    # -------------------------
    # TRAIN
    # -------------------------
    model.train()

    total_train_loss = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(x_batch)

        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # -------------------------
    # VALIDATION
    # -------------------------
    model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)

            loss = criterion(logits, y_batch)
            total_val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_accuracy"].append(val_accuracy)
    history["val_macro_f1"].append(val_macro_f1)

    # Save best model by Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_model_state = deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print("\nBest Validation Macro F1:", round(best_val_f1, 4))

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

for epoch in range(10, 20):

    model.train()
    total_train_loss = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            total_val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    scheduler.step(val_macro_f1)

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_model_state = deepcopy(model.state_dict())

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1:02d}/20 | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f} | "
        f"LR: {current_lr:.6f}"
    )

print("\nBest Validation Macro F1:", round(best_val_f1, 4))

In [ ]:
# Restore best TextCNN checkpoint
model.load_state_dict(best_model_state)
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x_batch, y_batch in val_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(x_batch)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

# Convert labels back from 0-4 to 1-5
all_preds_original = np.array(all_preds) + 1
all_labels_original = np.array(all_labels) + 1

print("TEXTCNN — BEST VALIDATION CHECKPOINT")
print()

print(
    classification_report(
        all_labels_original,
        all_preds_original,
        digits=4
    )
)

cm = confusion_matrix(
    all_labels_original,
    all_preds_original
)

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Neoplasms",
        "Digestive",
        "Nervous",
        "Cardiovascular",
        "General"
    ]
)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=30)
plt.title("TextCNN - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

train_labels_zero_based = (
    train_df["condition_label"].astype(int).values - 1
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(5),
    y=train_labels_zero_based
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class weights:", class_weights)

In [ ]:
torch.manual_seed(42)

weighted_model = TextCNN(
    vocab_size=len(vocab),
    embedding_dim=128,
    num_classes=5,
    num_filters=100,
    filter_sizes=(3, 4, 5),
    dropout=0.5
).to(device)

weighted_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

weighted_optimizer = torch.optim.Adam(
    weighted_model.parameters(),
    lr=1e-3
)

EPOCHS = 20

In [ ]:
from copy import deepcopy

best_weighted_f1 = -1
best_weighted_state = None

for epoch in range(EPOCHS):

    weighted_model.train()
    total_train_loss = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        weighted_optimizer.zero_grad()

        logits = weighted_model(x_batch)

        loss = weighted_criterion(
            logits,
            y_batch
        )

        loss.backward()
        weighted_optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    weighted_model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = weighted_model(x_batch)

            loss = weighted_criterion(
                logits,
                y_batch
            )

            total_val_loss += loss.item()

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                y_batch.cpu().numpy()
            )

    avg_val_loss = total_val_loss / len(val_loader)

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    if val_macro_f1 > best_weighted_f1:
        best_weighted_f1 = val_macro_f1
        best_weighted_state = deepcopy(
            weighted_model.state_dict()
        )

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print(
    "\nBest Weighted TextCNN Validation Macro F1:",
    round(best_weighted_f1, 4)
)

In [ ]:
weighted_model.load_state_dict(best_weighted_state)
weighted_model.eval()

weighted_preds = []
weighted_labels = []

with torch.no_grad():
    for x_batch, y_batch in val_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = weighted_model(x_batch)
        preds = torch.argmax(logits, dim=1)

        weighted_preds.extend(preds.cpu().numpy())
        weighted_labels.extend(y_batch.cpu().numpy())

weighted_preds_original = np.array(weighted_preds) + 1
weighted_labels_original = np.array(weighted_labels) + 1

print("WEIGHTED TEXTCNN — BEST VALIDATION CHECKPOINT\n")

print(
    classification_report(
        weighted_labels_original,
        weighted_preds_original,
        digits=4
    )
)

cm = confusion_matrix(
    weighted_labels_original,
    weighted_preds_original
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Neoplasms",
        "Digestive",
        "Nervous",
        "Cardiovascular",
        "General"
    ]
)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=30)
plt.title("Weighted TextCNN - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
torch.manual_seed(42)

cnn_variant = TextCNN(
    vocab_size=len(vocab),
    embedding_dim=128,
    num_classes=5,
    num_filters=100,
    filter_sizes=(2, 3, 4, 5),
    dropout=0.5
).to(device)

variant_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

variant_optimizer = torch.optim.Adam(
    cnn_variant.parameters(),
    lr=1e-3
)

print(cnn_variant)

print(
    "\nTrainable parameters:",
    f"{sum(p.numel() for p in cnn_variant.parameters() if p.requires_grad):,}"
)

In [ ]:
best_variant_f1 = -1
best_variant_state = None

for epoch in range(20):

    # TRAIN
    cnn_variant.train()
    total_train_loss = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        variant_optimizer.zero_grad()

        logits = cnn_variant(x_batch)

        loss = variant_criterion(
            logits,
            y_batch
        )

        loss.backward()
        variant_optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # VALIDATION
    cnn_variant.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = cnn_variant(x_batch)

            loss = variant_criterion(
                logits,
                y_batch
            )

            total_val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    if val_macro_f1 > best_variant_f1:
        best_variant_f1 = val_macro_f1
        best_variant_state = deepcopy(
            cnn_variant.state_dict()
        )

    print(
        f"Epoch {epoch+1:02d}/20 | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print(
    "\nBest CNN Variant Validation Macro F1:",
    round(best_variant_f1, 4)
)

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128,
        num_classes=5,
        num_layers=1,
        dropout=0.5,
        padding_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x):
        x = self.embedding(x)

        output, (hidden, cell) = self.lstm(x)

        # Last forward + last backward hidden states
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        x = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        x = self.dropout(x)

        return self.fc(x)

In [ ]:
# ==========================================================
# PHASE 3 — BiLSTM
# ==========================================================

## Objective
#Train a Bidirectional LSTM classifier to test whether sequential context
#and bidirectional information improve medical abstract classification
#compared with the classical ML and TextCNN baselines.

## Previous Best Results
#- Best Classical Model: TF-IDF + Balanced Logistic Regression
 # - Validation Macro F1: 0.8203

#- Best TextCNN:
 # - Weighted TextCNN
  #- Filter sizes: (2, 3, 4, 5)
   #- Validation Macro F1: 0.7863

## Main Evaluation Metric
#Macro F1

In [ ]:
torch.manual_seed(42)

bilstm_model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_classes=5,
    num_layers=1,
    dropout=0.5
).to(device)

print(bilstm_model)

total_params = sum(
    p.numel()
    for p in bilstm_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in bilstm_model.parameters()
    if p.requires_grad
)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# BiLSTM DataLoaders
BILSTM_BATCH_SIZE = 32

bilstm_train_loader = DataLoader(
    train_dataset,
    batch_size=BILSTM_BATCH_SIZE,
    shuffle=True
)

bilstm_val_loader = DataLoader(
    val_dataset,
    batch_size=BILSTM_BATCH_SIZE,
    shuffle=False
)

print("BiLSTM train batches:", len(bilstm_train_loader))
print("BiLSTM validation batches:", len(bilstm_val_loader))

In [ ]:
bilstm_criterion = nn.CrossEntropyLoss()

bilstm_optimizer = torch.optim.Adam(
    bilstm_model.parameters(),
    lr=1e-3
)

BILSTM_EPOCHS = 15

In [ ]:
from copy import deepcopy

best_bilstm_f1 = -1
best_bilstm_state = None

bilstm_history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": []
}

for epoch in range(BILSTM_EPOCHS):

    # --------------------
    # TRAIN
    # --------------------
    bilstm_model.train()
    total_train_loss = 0

    for x_batch, y_batch in bilstm_train_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        bilstm_optimizer.zero_grad()

        logits = bilstm_model(x_batch)

        loss = bilstm_criterion(
            logits,
            y_batch
        )

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            bilstm_model.parameters(),
            max_norm=1.0
        )

        bilstm_optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = (
        total_train_loss / len(bilstm_train_loader)
    )

    # --------------------
    # VALIDATION
    # --------------------
    bilstm_model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x_batch, y_batch in bilstm_val_loader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = bilstm_model(x_batch)

            loss = bilstm_criterion(
                logits,
                y_batch
            )

            total_val_loss += loss.item()

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                y_batch.cpu().numpy()
            )

    avg_val_loss = (
        total_val_loss / len(bilstm_val_loader)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    bilstm_history["train_loss"].append(
        avg_train_loss
    )

    bilstm_history["val_loss"].append(
        avg_val_loss
    )

    bilstm_history["val_accuracy"].append(
        val_accuracy
    )

    bilstm_history["val_macro_f1"].append(
        val_macro_f1
    )

    if val_macro_f1 > best_bilstm_f1:

        best_bilstm_f1 = val_macro_f1

        best_bilstm_state = deepcopy(
            bilstm_model.state_dict()
        )

    print(
        f"Epoch {epoch+1:02d}/{BILSTM_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print(
    "\nBest BiLSTM Validation Macro F1:",
    round(best_bilstm_f1, 4)
)

In [ ]:
class MedicalAbstractDatasetWithLength(Dataset):
    def __init__(self, dataframe, vocab, max_len):
        self.texts = dataframe["medical_abstract"].astype(str).tolist()
        self.labels = dataframe["condition_label"].astype(int).values - 1
        self.vocab = vocab
        self.max_len = max_len

    def encode(self, text):
        tokens = simple_tokenize(text)

        ids = [
            self.vocab.get(token, self.vocab["<UNK>"])
            for token in tokens
        ]

        ids = ids[:self.max_len]

        length = len(ids)

        if length < self.max_len:
            ids += [self.vocab["<PAD>"]] * (self.max_len - length)

        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(length, dtype=torch.long)
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x, length = self.encode(self.texts[idx])
        y = torch.tensor(self.labels[idx], dtype=torch.long)

        return x, length, y

In [ ]:
bilstm_train_dataset = MedicalAbstractDatasetWithLength(
    train_df, vocab, MAX_LEN
)

bilstm_val_dataset = MedicalAbstractDatasetWithLength(
    val_df, vocab, MAX_LEN
)

bilstm_train_loader = DataLoader(
    bilstm_train_dataset,
    batch_size=32,
    shuffle=True
)

bilstm_val_loader = DataLoader(
    bilstm_val_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128,
        num_classes=5,
        dropout=0.5,
        padding_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x, lengths):

        x = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        x = torch.cat(
            [forward_hidden, backward_hidden],
            dim=1
        )

        x = self.dropout(x)

        return self.fc(x)

In [ ]:
torch.manual_seed(42)

bilstm_model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_classes=5,
    dropout=0.5
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    bilstm_model.parameters(),
    lr=1e-3
)

In [ ]:
for x_batch, lengths, y_batch in bilstm_train_loader:

    x_batch = x_batch.to(device)
    lengths = lengths.to(device)
    y_batch = y_batch.to(device)

    logits = bilstm_model(x_batch, lengths)

In [ ]:
from copy import deepcopy

best_bilstm_f1 = -1
best_bilstm_state = None

BILSTM_EPOCHS = 15

for epoch in range(BILSTM_EPOCHS):

    # --------------------
    # TRAIN
    # --------------------
    bilstm_model.train()
    total_train_loss = 0

    for x_batch, lengths, y_batch in bilstm_train_loader:

        x_batch = x_batch.to(device)
        lengths = lengths.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = bilstm_model(
            x_batch,
            lengths
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            bilstm_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = (
        total_train_loss / len(bilstm_train_loader)
    )

    # --------------------
    # VALIDATION
    # --------------------
    bilstm_model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x_batch, lengths, y_batch in bilstm_val_loader:

            x_batch = x_batch.to(device)
            lengths = lengths.to(device)
            y_batch = y_batch.to(device)

            logits = bilstm_model(
                x_batch,
                lengths
            )

            loss = criterion(
                logits,
                y_batch
            )

            total_val_loss += loss.item()

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                y_batch.cpu().numpy()
            )

    avg_val_loss = (
        total_val_loss / len(bilstm_val_loader)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    if val_macro_f1 > best_bilstm_f1:
        best_bilstm_f1 = val_macro_f1

        best_bilstm_state = deepcopy(
            bilstm_model.state_dict()
        )

    print(
        f"Epoch {epoch+1:02d}/{BILSTM_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print(
    "\nBest Corrected BiLSTM Validation Macro F1:",
    round(best_bilstm_f1, 4)
)

In [ ]:
weighted_bilstm_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

torch.manual_seed(42)

weighted_bilstm_model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_classes=5,
    dropout=0.5
).to(device)

weighted_bilstm_optimizer = torch.optim.Adam(
    weighted_bilstm_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [ ]:
from copy import deepcopy

# Fresh weighted BiLSTM model
torch.manual_seed(42)

weighted_bilstm_model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    num_classes=5,
    dropout=0.5
).to(device)

weighted_bilstm_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

weighted_bilstm_optimizer = torch.optim.Adam(
    weighted_bilstm_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

WEIGHTED_BILSTM_EPOCHS = 10

best_weighted_bilstm_f1 = -1
best_weighted_bilstm_state = None

for epoch in range(WEIGHTED_BILSTM_EPOCHS):

    # --------------------
    # TRAIN
    # --------------------
    weighted_bilstm_model.train()
    total_train_loss = 0

    for x_batch, lengths, y_batch in bilstm_train_loader:

        x_batch = x_batch.to(device)
        lengths = lengths.to(device)
        y_batch = y_batch.to(device)

        weighted_bilstm_optimizer.zero_grad()

        logits = weighted_bilstm_model(
            x_batch,
            lengths
        )

        loss = weighted_bilstm_criterion(
            logits,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            weighted_bilstm_model.parameters(),
            max_norm=1.0
        )

        weighted_bilstm_optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = (
        total_train_loss / len(bilstm_train_loader)
    )

    # --------------------
    # VALIDATION
    # --------------------
    weighted_bilstm_model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x_batch, lengths, y_batch in bilstm_val_loader:

            x_batch = x_batch.to(device)
            lengths = lengths.to(device)
            y_batch = y_batch.to(device)

            logits = weighted_bilstm_model(
                x_batch,
                lengths
            )

            loss = weighted_bilstm_criterion(
                logits,
                y_batch
            )

            total_val_loss += loss.item()

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                y_batch.cpu().numpy()
            )

    avg_val_loss = (
        total_val_loss / len(bilstm_val_loader)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    if val_macro_f1 > best_weighted_bilstm_f1:
        best_weighted_bilstm_f1 = val_macro_f1
        best_weighted_bilstm_state = deepcopy(
            weighted_bilstm_model.state_dict()
        )

    print(
        f"Epoch {epoch+1:02d}/{WEIGHTED_BILSTM_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print(
    "\nBest Weighted BiLSTM Validation Macro F1:",
    round(best_weighted_bilstm_f1, 4)
)

In [ ]:
weighted_bilstm_model.load_state_dict(best_weighted_bilstm_state)
weighted_bilstm_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x_batch, lengths, y_batch in bilstm_val_loader:

        x_batch = x_batch.to(device)
        lengths = lengths.to(device)
        y_batch = y_batch.to(device)

        logits = weighted_bilstm_model(
            x_batch,
            lengths
        )

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

all_preds_original = np.array(all_preds) + 1
all_labels_original = np.array(all_labels) + 1

print("WEIGHTED BiLSTM — BEST VALIDATION CHECKPOINT\n")

print(
    classification_report(
        all_labels_original,
        all_preds_original,
        digits=4
    )
)

cm = confusion_matrix(
    all_labels_original,
    all_preds_original
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Neoplasms",
        "Digestive",
        "Nervous",
        "Cardiovascular",
        "General"
    ]
)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=30)
plt.title("Weighted BiLSTM - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# PHASE 6 — BERT
# ==========================================================

## Objective
#Fine-tune a pretrained BERT model for five-class medical abstract classification
#and compare contextual transformer representations against classical ML,
#TextCNN, BiLSTM, and DistilBERT.

## First Step
#Analyze BERT tokenizer sequence lengths before selecting max_length.

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from transformers import AutoTokenizer
import numpy as np

BERT_CHECKPOINT = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(
    BERT_CHECKPOINT
)

token_lengths = []

for text in train_df["medical_abstract"].astype(str):
    ids = bert_tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    token_lengths.append(len(ids))

token_lengths = np.array(token_lengths)

print("BERT TOKEN LENGTH ANALYSIS")
print("-" * 45)
print("Count:", len(token_lengths))
print("Mean:", round(token_lengths.mean(), 2))
print("Median:", np.median(token_lengths))
print("75th percentile:", np.percentile(token_lengths, 75))
print("90th percentile:", np.percentile(token_lengths, 90))
print("95th percentile:", np.percentile(token_lengths, 95))
print("97.5th percentile:", np.percentile(token_lengths, 97.5))
print("99th percentile:", np.percentile(token_lengths, 99))
print("Maximum:", token_lengths.max())

print("\nPotential truncation:")
print("-" * 45)

for limit in [128, 256, 384, 512]:
    count = (token_lengths > limit).sum()
    pct = count / len(token_lengths) * 100
    print(f"> {limit:3d} tokens: {count:4d} abstracts ({pct:.2f}%)")

In [ ]:
from torch.utils.data import Dataset

BERT_MAX_LENGTH = 512

class BERTMedicalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["medical_abstract"].astype(str).tolist()
        self.labels = dataframe["condition_label"].astype(int).values - 1
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

bert_train_dataset = BERTMedicalDataset(
    train_df,
    bert_tokenizer,
    BERT_MAX_LENGTH
)

bert_val_dataset = BERTMedicalDataset(
    val_df,
    bert_tokenizer,
    BERT_MAX_LENGTH
)

print("Train examples:", len(bert_train_dataset))
print("Validation examples:", len(bert_val_dataset))
print("Max sequence length:", BERT_MAX_LENGTH)

In [ ]:
from transformers import AutoModelForSequenceClassification

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_CHECKPOINT,
    num_labels=5
).to(device)

print("Model:", BERT_CHECKPOINT)
print("Device:", device)

total_params = sum(p.numel() for p in bert_model.parameters())
trainable_params = sum(
    p.numel() for p in bert_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
from torch.utils.data import DataLoader

BERT_BATCH_SIZE = 8

bert_train_loader = DataLoader(
    bert_train_dataset,
    batch_size=BERT_BATCH_SIZE,
    shuffle=True
)

bert_val_loader = DataLoader(
    bert_val_dataset,
    batch_size=BERT_BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(bert_train_loader))
print("Validation batches:", len(bert_val_loader))

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from copy import deepcopy
import torch

BERT_EPOCHS = 3

bert_optimizer = AdamW(
    bert_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

total_training_steps = (
    len(bert_train_loader) * BERT_EPOCHS
)

warmup_steps = int(
    0.1 * total_training_steps
)

bert_scheduler = get_linear_schedule_with_warmup(
    bert_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

best_bert_f1 = -1
best_bert_state = None

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

print("Epochs:", BERT_EPOCHS)
print("Learning rate:", 2e-5)
print("Warmup steps:", warmup_steps)
print("Total training steps:", total_training_steps)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

for epoch in range(BERT_EPOCHS):

    # -------------------------
    # TRAIN
    # -------------------------
    bert_model.train()

    total_train_loss = 0

    for batch in bert_train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        bert_optimizer.zero_grad()

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):
            outputs = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()

        scaler.unscale_(bert_optimizer)

        torch.nn.utils.clip_grad_norm_(
            bert_model.parameters(),
            max_norm=1.0
        )

        scaler.step(bert_optimizer)
        scaler.update()

        bert_scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = (
        total_train_loss / len(bert_train_loader)
    )

    # -------------------------
    # VALIDATION
    # -------------------------
    bert_model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in bert_val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.amp.autocast(
                "cuda",
                enabled=torch.cuda.is_available()
            ):
                outputs = bert_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            total_val_loss += outputs.loss.item()

            preds = torch.argmax(
                outputs.logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_val_loss = (
        total_val_loss / len(bert_val_loader)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    if val_macro_f1 > best_bert_f1:

        best_bert_f1 = val_macro_f1

        best_bert_state = deepcopy(
            bert_model.state_dict()
        )

    print(
        f"Epoch {epoch+1:02d}/{BERT_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print(
    "\nBest BERT Validation Macro F1:",
    round(best_bert_f1, 4)
)

In [ ]:
bert_model.load_state_dict(best_bert_state)
bert_model.eval()

bert_preds = []
bert_labels = []

with torch.no_grad():
    for batch in bert_val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        bert_preds.extend(
            preds.cpu().numpy()
        )

        bert_labels.extend(
            labels.cpu().numpy()
        )

bert_preds_original = np.array(bert_preds) + 1
bert_labels_original = np.array(bert_labels) + 1

print("BERT — BEST VALIDATION CHECKPOINT\n")

print(
    classification_report(
        bert_labels_original,
        bert_preds_original,
        digits=4
    )
)

cm = confusion_matrix(
    bert_labels_original,
    bert_preds_original
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Neoplasms",
        "Digestive",
        "Nervous",
        "Cardiovascular",
        "General"
    ]
)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=30)

plt.title("BERT - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# PHASE 7 — BioBERT
# ==========================================================

## Objective
#Test whether biomedical-domain pretraining improves medical abstract
#classification compared with the general BERT baseline.

## Comparison Baseline
#BERT validation Macro F1: 0.8124

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BIOBERT_CHECKPOINT = "dmis-lab/biobert-base-cased-v1.2"

biobert_tokenizer = AutoTokenizer.from_pretrained(
    BIOBERT_CHECKPOINT
)

biobert_model = AutoModelForSequenceClassification.from_pretrained(
    BIOBERT_CHECKPOINT,
    num_labels=5
).to(device)

print("Model:", BIOBERT_CHECKPOINT)
print("Device:", device)

total_params = sum(p.numel() for p in biobert_model.parameters())
trainable_params = sum(
    p.numel() for p in biobert_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
biobert_lengths = []

for text in train_df["medical_abstract"].astype(str):
    ids = biobert_tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    biobert_lengths.append(len(ids))

biobert_lengths = np.array(biobert_lengths)

print("BIOBERT TOKEN LENGTH ANALYSIS")
print("-" * 45)
print("Mean:", round(biobert_lengths.mean(), 2))
print("Median:", np.median(biobert_lengths))
print("90th percentile:", np.percentile(biobert_lengths, 90))
print("95th percentile:", np.percentile(biobert_lengths, 95))
print("99th percentile:", np.percentile(biobert_lengths, 99))
print("Maximum:", biobert_lengths.max())

for limit in [256, 384, 512]:
    count = (biobert_lengths > limit).sum()
    pct = count / len(biobert_lengths) * 100
    print(f"> {limit} tokens: {count} abstracts ({pct:.2f}%)")

In [ ]:
print("BioBERT num labels:", biobert_model.config.num_labels)

In [ ]:
BIOBERT_MAX_LENGTH = 512

class BioBERTMedicalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["medical_abstract"].astype(str).tolist()
        self.labels = dataframe["condition_label"].astype(int).values - 1
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

biobert_train_dataset = BioBERTMedicalDataset(
    train_df,
    biobert_tokenizer,
    BIOBERT_MAX_LENGTH
)

biobert_val_dataset = BioBERTMedicalDataset(
    val_df,
    biobert_tokenizer,
    BIOBERT_MAX_LENGTH
)

print("Train examples:", len(biobert_train_dataset))
print("Validation examples:", len(biobert_val_dataset))
print("Max sequence length:", BIOBERT_MAX_LENGTH)

In [ ]:
BIOBERT_BATCH_SIZE = 8

biobert_train_loader = DataLoader(
    biobert_train_dataset,
    batch_size=BIOBERT_BATCH_SIZE,
    shuffle=True
)

biobert_val_loader = DataLoader(
    biobert_val_dataset,
    batch_size=BIOBERT_BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(biobert_train_loader))
print("Validation batches:", len(biobert_val_loader))

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from copy import deepcopy
from sklearn.metrics import accuracy_score, f1_score
import torch

BIOBERT_EPOCHS = 3

biobert_optimizer = AdamW(
    biobert_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

total_training_steps = (
    len(biobert_train_loader) * BIOBERT_EPOCHS
)

warmup_steps = int(
    0.1 * total_training_steps
)

biobert_scheduler = get_linear_schedule_with_warmup(
    biobert_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

best_biobert_f1 = -1
best_biobert_state = None

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

print("Epochs:", BIOBERT_EPOCHS)
print("Learning rate:", 2e-5)
print("Warmup steps:", warmup_steps)
print("Total training steps:", total_training_steps)

In [ ]:
for epoch in range(BIOBERT_EPOCHS):

    # -------------------------
    # TRAIN
    # -------------------------
    biobert_model.train()
    total_train_loss = 0

    for batch in biobert_train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        biobert_optimizer.zero_grad()

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):
            outputs = biobert_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()

        scaler.unscale_(biobert_optimizer)

        torch.nn.utils.clip_grad_norm_(
            biobert_model.parameters(),
            max_norm=1.0
        )

        scaler.step(biobert_optimizer)
        scaler.update()

        biobert_scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = (
        total_train_loss / len(biobert_train_loader)
    )

    # -------------------------
    # VALIDATION
    # -------------------------
    biobert_model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in biobert_val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.amp.autocast(
                "cuda",
                enabled=torch.cuda.is_available()
            ):
                outputs = biobert_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            total_val_loss += outputs.loss.item()

            preds = torch.argmax(
                outputs.logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_val_loss = (
        total_val_loss / len(biobert_val_loader)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    if val_macro_f1 > best_biobert_f1:

        best_biobert_f1 = val_macro_f1

        best_biobert_state = deepcopy(
            biobert_model.state_dict()
        )

    print(
        f"Epoch {epoch+1:02d}/{BIOBERT_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print(
    "\nBest BioBERT Validation Macro F1:",
    round(best_biobert_f1, 4)
)

In [ ]:
biobert_model.load_state_dict(best_biobert_state)
biobert_model.eval()

biobert_preds = []
biobert_labels = []

with torch.no_grad():
    for batch in biobert_val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = biobert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        biobert_preds.extend(
            preds.cpu().numpy()
        )

        biobert_labels.extend(
            labels.cpu().numpy()
        )

biobert_preds_original = np.array(biobert_preds) + 1
biobert_labels_original = np.array(biobert_labels) + 1

print("BIOBERT — BEST VALIDATION CHECKPOINT\n")

print(
    classification_report(
        biobert_labels_original,
        biobert_preds_original,
        digits=4
    )
)

cm = confusion_matrix(
    biobert_labels_original,
    biobert_preds_original
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Neoplasms",
        "Digestive",
        "Nervous",
        "Cardiovascular",
        "General"
    ]
)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=30)

plt.title("BioBERT - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
#biomert

In [ ]:
BIOMEDBERT_CHECKPOINT = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BIOMEDBERT_CHECKPOINT = (
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

biomedbert_tokenizer = AutoTokenizer.from_pretrained(
    BIOMEDBERT_CHECKPOINT
)

biomedbert_model = AutoModelForSequenceClassification.from_pretrained(
    BIOMEDBERT_CHECKPOINT,
    num_labels=5
).to(device)

print("Model:", BIOMEDBERT_CHECKPOINT)
print("Device:", device)

total_params = sum(
    p.numel()
    for p in biomedbert_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in biomedbert_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
biomedbert_lengths = []

for text in train_df["medical_abstract"].astype(str):
    ids = biomedbert_tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    biomedbert_lengths.append(len(ids))

biomedbert_lengths = np.array(biomedbert_lengths)

print("BIOMEDBERT TOKEN LENGTH ANALYSIS")
print("-" * 45)

print("Count:", len(biomedbert_lengths))
print("Mean:", round(biomedbert_lengths.mean(), 2))
print("Median:", np.median(biomedbert_lengths))
print("75th percentile:", np.percentile(biomedbert_lengths, 75))
print("90th percentile:", np.percentile(biomedbert_lengths, 90))
print("95th percentile:", np.percentile(biomedbert_lengths, 95))
print("97.5th percentile:", np.percentile(biomedbert_lengths, 97.5))
print("99th percentile:", np.percentile(biomedbert_lengths, 99))
print("Maximum:", biomedbert_lengths.max())

print("\nPotential truncation:")
print("-" * 45)

for limit in [256, 384, 512]:
    count = (biomedbert_lengths > limit).sum()
    pct = count / len(biomedbert_lengths) * 100

    print(
        f"> {limit:3d} tokens: "
        f"{count:4d} abstracts "
        f"({pct:.2f}%)"
    )

In [ ]:
BIOMEDBERT_MAX_LENGTH = 512

class BiomedBERTMedicalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["medical_abstract"].astype(str).tolist()
        self.labels = dataframe["condition_label"].astype(int).values - 1
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

biomedbert_train_dataset = BiomedBERTMedicalDataset(
    train_df,
    biomedbert_tokenizer,
    BIOMEDBERT_MAX_LENGTH
)

biomedbert_val_dataset = BiomedBERTMedicalDataset(
    val_df,
    biomedbert_tokenizer,
    BIOMEDBERT_MAX_LENGTH
)

BIOMEDBERT_BATCH_SIZE = 8

biomedbert_train_loader = DataLoader(
    biomedbert_train_dataset,
    batch_size=BIOMEDBERT_BATCH_SIZE,
    shuffle=True
)

biomedbert_val_loader = DataLoader(
    biomedbert_val_dataset,
    batch_size=BIOMEDBERT_BATCH_SIZE,
    shuffle=False
)

print("Train examples:", len(biomedbert_train_dataset))
print("Validation examples:", len(biomedbert_val_dataset))
print("Train batches:", len(biomedbert_train_loader))
print("Validation batches:", len(biomedbert_val_loader))
print("Max sequence length:", BIOMEDBERT_MAX_LENGTH)

In [ ]:
BIOMEDBERT_EPOCHS = 3

biomedbert_optimizer = AdamW(
    biomedbert_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

total_training_steps = (
    len(biomedbert_train_loader) * BIOMEDBERT_EPOCHS
)

warmup_steps = int(
    0.1 * total_training_steps
)

biomedbert_scheduler = get_linear_schedule_with_warmup(
    biomedbert_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

best_biomedbert_f1 = -1
best_biomedbert_state = None

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

print("Epochs:", BIOMEDBERT_EPOCHS)
print("Learning rate:", 2e-5)
print("Warmup steps:", warmup_steps)
print("Total training steps:", total_training_steps)

In [ ]:
for epoch in range(BIOMEDBERT_EPOCHS):

    # -------------------------
    # TRAIN
    # -------------------------
    biomedbert_model.train()
    total_train_loss = 0

    for batch in biomedbert_train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        biomedbert_optimizer.zero_grad()

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):
            outputs = biomedbert_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()

        scaler.unscale_(biomedbert_optimizer)

        torch.nn.utils.clip_grad_norm_(
            biomedbert_model.parameters(),
            max_norm=1.0
        )

        scaler.step(biomedbert_optimizer)
        scaler.update()

        biomedbert_scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = (
        total_train_loss / len(biomedbert_train_loader)
    )

    # -------------------------
    # VALIDATION
    # -------------------------
    biomedbert_model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in biomedbert_val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.amp.autocast(
                "cuda",
                enabled=torch.cuda.is_available()
            ):
                outputs = biomedbert_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            total_val_loss += outputs.loss.item()

            preds = torch.argmax(
                outputs.logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_val_loss = (
        total_val_loss / len(biomedbert_val_loader)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    if val_macro_f1 > best_biomedbert_f1:

        best_biomedbert_f1 = val_macro_f1

        best_biomedbert_state = deepcopy(
            biomedbert_model.state_dict()
        )

    print(
        f"Epoch {epoch+1:02d}/{BIOMEDBERT_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print(
    "\nBest BiomedBERT Validation Macro F1:",
    round(best_biomedbert_f1, 4)
)

In [ ]:
biomedbert_model.load_state_dict(best_biomedbert_state)
biomedbert_model.eval()

biomedbert_preds = []
biomedbert_labels = []

with torch.no_grad():
    for batch in biomedbert_val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = biomedbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        biomedbert_preds.extend(
            preds.cpu().numpy()
        )

        biomedbert_labels.extend(
            labels.cpu().numpy()
        )

biomedbert_preds_original = np.array(biomedbert_preds) + 1
biomedbert_labels_original = np.array(biomedbert_labels) + 1

print("BIOMEDBERT — BEST VALIDATION CHECKPOINT\n")

print(
    classification_report(
        biomedbert_labels_original,
        biomedbert_preds_original,
        digits=4
    )
)

cm = confusion_matrix(
    biomedbert_labels_original,
    biomedbert_preds_original
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Neoplasms",
        "Digestive",
        "Nervous",
        "Cardiovascular",
        "General"
    ]
)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=30)

plt.title("BiomedBERT - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
batch = next(iter(val_loader))

print("Batch type:", type(batch))

if isinstance(batch, dict):
    print("Keys:", batch.keys())
else:
    print("Length:", len(batch))
    print("Item types:", [type(x) for x in batch])

In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score

biomedbert_model.eval()

all_probs = []
all_preds = []
all_true = []

with torch.no_grad():
    for batch in val_loader_5class:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = biomedbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_true = np.array(all_true)

confidences = all_probs.max(axis=1)

print("Validation Accuracy:", accuracy_score(all_true, all_preds))
print("Mean Confidence:", confidences.mean())

In [ ]:
from sklearn.metrics import accuracy_score

thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]

for t in thresholds:
    mask = confidences >= t

    coverage = mask.mean()

    if mask.sum() > 0:
        acc = accuracy_score(
            all_true[mask],
            all_preds[mask]
        )

        print(
            f"Threshold {t:.1f} | "
            f"Coverage: {coverage:.2%} | "
            f"Accepted: {mask.sum()} | "
            f"Accuracy: {acc:.4f}"
        )

In [ ]:
def expected_calibration_error(y_true, y_pred, confidences, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])

        if mask.sum() > 0:
            bin_acc = np.mean(y_true[mask] == y_pred[mask])
            bin_conf = np.mean(confidences[mask])

            ece += (mask.sum() / len(y_true)) * abs(bin_acc - bin_conf)

    return ece

ece = expected_calibration_error(
    all_true,
    all_preds,
    confidences
)

print("ECE:", round(ece, 4))

In [ ]:
import time
import torch
import numpy as np

biomedbert_model.eval()

# Warm-up
batch = next(iter(val_loader_5class))
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

with torch.no_grad():
    _ = biomedbert_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

torch.cuda.synchronize()

# Measure full validation inference time
start = time.time()

with torch.no_grad():
    for batch in val_loader_5class:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        _ = biomedbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

torch.cuda.synchronize()

elapsed = time.time() - start

num_examples = len(val_loader_5class.dataset)

print("Validation examples:", num_examples)
print("Total inference time (s):", round(elapsed, 2))
print("Milliseconds per abstract:", round((elapsed / num_examples) * 1000, 2))
print("Abstracts per second:", round(num_examples / elapsed, 2))

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# Load locked test set
test_df = pd.read_csv("test.csv")

class TestDataset(Dataset):
    def __init__(self, df):
        self.texts = df["medical_abstract"].astype(str).tolist()
        self.labels = (df["condition_label"] - 1).tolist()  # 1-5 -> 0-4

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

test_dataset = TestDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("Test examples:", len(test_dataset))

In [ ]:
biomedbert_model.eval()

test_preds = []
test_true = []
test_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = biomedbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        test_probs.extend(probs.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())
        test_true.extend(labels.cpu().numpy())

test_preds = np.array(test_preds)
test_true = np.array(test_true)
test_probs = np.array(test_probs)

accuracy = accuracy_score(test_true, test_preds)

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    test_true,
    test_preds,
    average="macro",
    zero_division=0
)

weighted_f1 = precision_recall_fscore_support(
    test_true,
    test_preds,
    average="weighted",
    zero_division=0
)[2]

print("FINAL TEST RESULTS")
print("------------------")
print("Accuracy:", round(accuracy, 4))
print("Macro Precision:", round(macro_precision, 4))
print("Macro Recall:", round(macro_recall, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted F1:", round(weighted_f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        test_true,
        test_preds,
        target_names=[
            "Neoplasms",
            "Digestive",
            "Nervous",
            "Cardiovascular",
            "General"
        ],
        digits=4
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(test_true, test_preds))

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class FiveClassDataset(Dataset):
    def __init__(self, df):
        self.texts = df["medical_abstract"].astype(str).tolist()

        # Original classes 1-5 -> model labels 0-4
        self.labels = (df["condition_label"] - 1).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

val_5class_dataset = FiveClassDataset(val_df)

val_loader_5class = DataLoader(
    val_5class_dataset,
    batch_size=8,
    shuffle=False
)

labels_check = []

for batch in val_loader_5class:
    labels_check.extend(batch["labels"].numpy())

print("Unique labels:", sorted(set(labels_check)))
print("Examples:", len(labels_check))

In [ ]:
# ==========================================================
# PHASE 9 — DeBERTa-v3
# ==========================================================

## Objective
#Evaluate whether a modern transformer architecture can outperform
#BERT and compete with biomedical-specific transformers.

## Current Best
#BiomedBERT Validation Macro F1: 0.8672

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEBERTA_CHECKPOINT = "microsoft/deberta-v3-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(
    DEBERTA_CHECKPOINT
)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_CHECKPOINT,
    num_labels=5
).to(device)

print("Model:", DEBERTA_CHECKPOINT)
print("Device:", device)

total_params = sum(
    p.numel()
    for p in deberta_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in deberta_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
deberta_lengths = []

for text in train_df["medical_abstract"].astype(str):

    ids = deberta_tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    deberta_lengths.append(len(ids))

deberta_lengths = np.array(deberta_lengths)

print("DEBERTA TOKEN LENGTH ANALYSIS")
print("-" * 45)

print("Count:", len(deberta_lengths))
print("Mean:", round(deberta_lengths.mean(), 2))
print("Median:", np.median(deberta_lengths))
print("75th percentile:", np.percentile(deberta_lengths, 75))
print("90th percentile:", np.percentile(deberta_lengths, 90))
print("95th percentile:", np.percentile(deberta_lengths, 95))
print("97.5th percentile:", np.percentile(deberta_lengths, 97.5))
print("99th percentile:", np.percentile(deberta_lengths, 99))
print("Maximum:", deberta_lengths.max())

print("\nPotential truncation:")
print("-" * 45)

for limit in [256, 384, 512]:
    count = (deberta_lengths > limit).sum()
    pct = count / len(deberta_lengths) * 100

    print(
        f"> {limit:3d} tokens: "
        f"{count:4d} abstracts "
        f"({pct:.2f}%)"
    )

In [ ]:
DEBERTA_MAX_LENGTH = 512

class DeBERTaMedicalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["medical_abstract"].astype(str).tolist()
        self.labels = dataframe["condition_label"].astype(int).values - 1
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

deberta_train_dataset = DeBERTaMedicalDataset(
    train_df,
    deberta_tokenizer,
    DEBERTA_MAX_LENGTH
)

deberta_val_dataset = DeBERTaMedicalDataset(
    val_df,
    deberta_tokenizer,
    DEBERTA_MAX_LENGTH
)

DEBERTA_BATCH_SIZE = 8

deberta_train_loader = DataLoader(
    deberta_train_dataset,
    batch_size=DEBERTA_BATCH_SIZE,
    shuffle=True
)

deberta_val_loader = DataLoader(
    deberta_val_dataset,
    batch_size=DEBERTA_BATCH_SIZE,
    shuffle=False
)

print("Train examples:", len(deberta_train_dataset))
print("Validation examples:", len(deberta_val_dataset))
print("Train batches:", len(deberta_train_loader))
print("Validation batches:", len(deberta_val_loader))
print("Max sequence length:", DEBERTA_MAX_LENGTH)

In [ ]:
DEBERTA_EPOCHS = 3

deberta_optimizer = AdamW(
    deberta_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

total_training_steps = (
    len(deberta_train_loader) * DEBERTA_EPOCHS
)

warmup_steps = int(
    0.1 * total_training_steps
)

deberta_scheduler = get_linear_schedule_with_warmup(
    deberta_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

best_deberta_f1 = -1
best_deberta_state = None

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

print("Epochs:", DEBERTA_EPOCHS)
print("Learning rate:", 2e-5)
print("Warmup steps:", warmup_steps)
print("Total training steps:", total_training_steps)

In [ ]:
print("Model parameter dtype:", next(deberta_model.parameters()).dtype)

In [ ]:
# Force model parameters to FP32
deberta_model = deberta_model.float().to(device)

print("New model dtype:", next(deberta_model.parameters()).dtype)

DEBERTA_EPOCHS = 3

deberta_optimizer = AdamW(
    deberta_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

total_training_steps = (
    len(deberta_train_loader) * DEBERTA_EPOCHS
)

warmup_steps = int(0.1 * total_training_steps)

deberta_scheduler = get_linear_schedule_with_warmup(
    deberta_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

best_deberta_f1 = -1
best_deberta_state = None

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

print("Ready.")
print("Epochs:", DEBERTA_EPOCHS)
print("Learning rate:", 2e-5)
print("Warmup steps:", warmup_steps)
print("Total training steps:", total_training_steps)

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

DEBERTA_BATCH_SIZE = 4

deberta_train_loader = DataLoader(
    deberta_train_dataset,
    batch_size=DEBERTA_BATCH_SIZE,
    shuffle=True
)

deberta_val_loader = DataLoader(
    deberta_val_dataset,
    batch_size=DEBERTA_BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(deberta_train_loader))
print("Validation batches:", len(deberta_val_loader))

In [ ]:
DEBERTA_EPOCHS = 3

deberta_optimizer = AdamW(
    deberta_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

total_training_steps = (
    len(deberta_train_loader) * DEBERTA_EPOCHS
)

warmup_steps = int(0.1 * total_training_steps)

deberta_scheduler = get_linear_schedule_with_warmup(
    deberta_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

best_deberta_f1 = -1
best_deberta_state = None

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

print("Batch size:", DEBERTA_BATCH_SIZE)
print("Warmup steps:", warmup_steps)
print("Total training steps:", total_training_steps)

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared.")

In [ ]:
del bert_model
del biobert_model
del biomedbert_model

gc.collect()
torch.cuda.empty_cache()

print("Old transformer models deleted and GPU memory cleared.")

In [ ]:
print(torch.cuda.memory_summary(device=None, abbreviated=True))

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

DEBERTA_BATCH_SIZE = 2

deberta_train_loader = DataLoader(
    deberta_train_dataset,
    batch_size=DEBERTA_BATCH_SIZE,
    shuffle=True
)

deberta_val_loader = DataLoader(
    deberta_val_dataset,
    batch_size=DEBERTA_BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(deberta_train_loader))
print("Validation batches:", len(deberta_val_loader))

In [ ]:
import gc
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from copy import deepcopy
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DEBERTA_CHECKPOINT = "microsoft/deberta-v3-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(
    DEBERTA_CHECKPOINT
)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_CHECKPOINT,
    num_labels=5
)

# Important: keep parameters FP32
deberta_model = deberta_model.float().to(device)

print("Model:", DEBERTA_CHECKPOINT)
print("Model dtype:", next(deberta_model.parameters()).dtype)
print("Device:", next(deberta_model.parameters()).device)

In [ ]:
DEBERTA_MAX_LENGTH = 512

class DeBERTaMedicalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["medical_abstract"].astype(str).tolist()
        self.labels = (
            dataframe["condition_label"].astype(int).values - 1
        )
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

deberta_train_dataset = DeBERTaMedicalDataset(
    train_df,
    deberta_tokenizer,
    DEBERTA_MAX_LENGTH
)

deberta_val_dataset = DeBERTaMedicalDataset(
    val_df,
    deberta_tokenizer,
    DEBERTA_MAX_LENGTH
)

In [ ]:
DEBERTA_BATCH_SIZE = 2

deberta_train_loader = DataLoader(
    deberta_train_dataset,
    batch_size=DEBERTA_BATCH_SIZE,
    shuffle=True
)

deberta_val_loader = DataLoader(
    deberta_val_dataset,
    batch_size=DEBERTA_BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(deberta_train_loader))
print("Validation batches:", len(deberta_val_loader))

In [ ]:
DEBERTA_EPOCHS = 3
GRAD_ACCUM_STEPS = 4

deberta_optimizer = AdamW(
    deberta_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

updates_per_epoch = int(
    np.ceil(
        len(deberta_train_loader) / GRAD_ACCUM_STEPS
    )
)

total_training_steps = (
    updates_per_epoch * DEBERTA_EPOCHS
)

warmup_steps = int(
    0.1 * total_training_steps
)

deberta_scheduler = get_linear_schedule_with_warmup(
    deberta_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

best_deberta_f1 = -1
best_deberta_state = None

print("Physical batch size:", DEBERTA_BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)
print(
    "Effective batch size:",
    DEBERTA_BATCH_SIZE * GRAD_ACCUM_STEPS
)
print("Epochs:", DEBERTA_EPOCHS)
print("Total optimizer steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

In [ ]:
for epoch in range(DEBERTA_EPOCHS):

    # ========================
    # TRAIN
    # ========================

    deberta_model.train()

    total_train_loss = 0

    deberta_optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(deberta_train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16
        ):

            outputs = deberta_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            raw_loss = outputs.loss

            loss = raw_loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        total_train_loss += raw_loss.item()

        should_update = (
            (step + 1) % GRAD_ACCUM_STEPS == 0
            or
            (step + 1) == len(deberta_train_loader)
        )

        if should_update:

            scaler.unscale_(deberta_optimizer)

            torch.nn.utils.clip_grad_norm_(
                deberta_model.parameters(),
                max_norm=1.0
            )

            scaler.step(deberta_optimizer)
            scaler.update()

            deberta_optimizer.zero_grad(
                set_to_none=True
            )

            deberta_scheduler.step()

    avg_train_loss = (
        total_train_loss /
        len(deberta_train_loader)
    )

    # ========================
    # VALIDATION
    # ========================

    deberta_model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in deberta_val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16
            ):

                outputs = deberta_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            total_val_loss += outputs.loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            all_preds.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_val_loss = (
        total_val_loss /
        len(deberta_val_loader)
    )

    val_accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    if val_macro_f1 > best_deberta_f1:

        best_deberta_f1 = val_macro_f1

        best_deberta_state = deepcopy(
            deberta_model.state_dict()
        )

    print(
        f"Epoch {epoch+1:02d}/{DEBERTA_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_macro_f1:.4f}"
    )

print(
    "\nBest DeBERTa Validation Macro F1:",
    round(best_deberta_f1, 4)
)

In [ ]:
deberta_model.load_state_dict(best_deberta_state)
deberta_model.eval()

deberta_preds = []
deberta_labels = []

with torch.no_grad():
    for batch in deberta_val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.amp.autocast(
            "cuda",
            dtype=torch.float16
        ):
            outputs = deberta_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        deberta_preds.extend(
            preds.cpu().numpy()
        )

        deberta_labels.extend(
            labels.cpu().numpy()
        )

deberta_preds_original = np.array(deberta_preds) + 1
deberta_labels_original = np.array(deberta_labels) + 1

print("DEBERTA-v3 — BEST VALIDATION CHECKPOINT\n")

print(
    classification_report(
        deberta_labels_original,
        deberta_preds_original,
        digits=4
    )
)

cm = confusion_matrix(
    deberta_labels_original,
    deberta_preds_original
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Neoplasms",
        "Digestive",
        "Nervous",
        "Cardiovascular",
        "General"
    ]
)

fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=30)

plt.title("DeBERTa-v3 - Validation Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# FINAL VALIDATION BENCHMARK
# ==========================================================

#All models below were evaluated on the same validation split.

#Primary metric: Macro F1
#Secondary metric: Accuracy

In [ ]:
import pandas as pd
import numpy as np

benchmark_results = pd.DataFrame([
    {
        "Model": "Multinomial Naive Bayes",
        "Family": "Classical ML",
        "Accuracy": 0.675502,
        "Macro_F1": 0.484887,
        "Weighted_F1": 0.614248
    },

    {
        "Model": "Logistic Regression",
        "Family": "Classical ML",
        "Accuracy": 0.821687,
        "Macro_F1": 0.820326,
        "Weighted_F1": 0.821105
    },

    {
        "Model": "Linear SVM",
        "Family": "Classical ML",
        "Accuracy": 0.824096,
        "Macro_F1": 0.817993,
        "Weighted_F1": 0.823911
    },

    {
        "Model": "TextCNN",
        "Family": "Neural",
        "Accuracy": 0.7871,
        "Macro_F1": 0.7882,
        "Weighted_F1": np.nan
    },

    {
        "Model": "Weighted BiLSTM",
        "Family": "Neural",
        "Accuracy": 0.7333,
        "Macro_F1": 0.7127,
        "Weighted_F1": 0.7327
    },

    {
        "Model": "DistilBERT",
        "Family": "Transformer",
        "Accuracy": np.nan,
        "Macro_F1": 0.8027,
        "Weighted_F1": np.nan
    },

    {
        "Model": "BERT-base",
        "Family": "Transformer",
        "Accuracy": 0.8153,
        "Macro_F1": 0.8124,
        "Weighted_F1": 0.8138
    },

    {
        "Model": "BioBERT",
        "Family": "Biomedical Transformer",
        "Accuracy": 0.8490,
        "Macro_F1": 0.8456,
        "Weighted_F1": 0.8483
    },

    {
        "Model": "BiomedBERT",
        "Family": "Biomedical Transformer",
        "Accuracy": 0.8683,
        "Macro_F1": 0.8672,
        "Weighted_F1": 0.8665
    },

    {
        "Model": "DeBERTa-v3",
        "Family": "Modern Transformer",
        "Accuracy": 0.8313,
        "Macro_F1": 0.8322,
        "Weighted_F1": np.nan
    }
])

benchmark_results = benchmark_results.sort_values(
    by="Macro_F1",
    ascending=False
).reset_index(drop=True)

benchmark_results.insert(
    0,
    "Rank",
    range(1, len(benchmark_results) + 1)
)

display(benchmark_results)

benchmark_results.to_csv(
    "final_validation_benchmark.csv",
    index=False
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

per_class_f1 = pd.DataFrame({
    "Model": [
        "BiomedBERT",
        "BioBERT",
        "DeBERTa-v3",
        "Logistic Regression",
        "BERT-base",
        "Weighted BiLSTM"
    ],

    "Neoplasms": [
        0.9235,
        0.9080,
        0.9030,
        0.8865,
        0.8909,
        0.8479
    ],

    "Digestive": [
        0.8826,
        0.8558,
        0.8720,
        0.8282,
        0.8311,
        0.6609
    ],

    "Nervous": [
        0.8150,
        0.7864,
        0.7862,
        0.7802,
        0.7476,
        0.5926
    ],

    "Cardiovascular": [
        0.9228,
        0.9069,
        0.8684,
        0.8682,
        0.8769,
        0.8432
    ],

    "General": [
        0.7859,
        0.7708,
        0.7311,
        0.7385,
        0.7155,
        0.6191
    ]
})

display(per_class_f1)

In [ ]:
import numpy as np

heatmap_df = per_class_f1.set_index("Model")

fig, ax = plt.subplots(figsize=(11, 6))

im = ax.imshow(
    heatmap_df.values,
    aspect="auto"
)

ax.set_xticks(
    np.arange(len(heatmap_df.columns))
)

ax.set_xticklabels(
    heatmap_df.columns
)

ax.set_yticks(
    np.arange(len(heatmap_df.index))
)

ax.set_yticklabels(
    heatmap_df.index
)

for i in range(len(heatmap_df.index)):
    for j in range(len(heatmap_df.columns)):
        ax.text(
            j,
            i,
            f"{heatmap_df.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.colorbar(
    im,
    label="F1 Score"
)

plt.title(
    "Validation F1 per Class — Verified Model Results"
)

plt.xlabel("Disease Class")
plt.ylabel("Model")

plt.tight_layout()
plt.show()

In [ ]:
benchmark_results.loc[
    benchmark_results["Model"] == "DeBERTa-v3",
    "Weighted_F1"
] = 0.827843

benchmark_results.loc[
    benchmark_results["Model"] == "Linear SVM",
    ["Accuracy", "Macro_F1", "Weighted_F1"]
] = [0.824096, 0.817993, 0.823911]

benchmark_results.loc[
    benchmark_results["Model"] == "Logistic Regression",
    ["Accuracy", "Macro_F1", "Weighted_F1"]
] = [0.821687, 0.820326, 0.821105]

display(benchmark_results)

In [ ]:
benchmark_results.loc[
    benchmark_results["Model"] == "DeBERTa-v3",
    "Weighted_F1"
] = 0.827843

benchmark_results.loc[
    benchmark_results["Model"] == "Linear SVM",
    ["Accuracy", "Macro_F1", "Weighted_F1"]
] = [0.824096, 0.817993, 0.823911]

benchmark_results.loc[
    benchmark_results["Model"] == "Logistic Regression",
    ["Accuracy", "Macro_F1", "Weighted_F1"]
] = [0.821687, 0.820326, 0.821105]

display(benchmark_results)

In [ ]:
for name in ["cnn_variant", "best_variant_state", "val_loader"]:
    print(
        name,
        "FOUND" if name in globals() else "MISSING"
    )

In [ ]:
analysis_models = per_class_f1.copy()

analysis_models["General_Gap"] = (
    analysis_models[
        ["Neoplasms", "Digestive", "Nervous", "Cardiovascular"]
    ].mean(axis=1)
    - analysis_models["General"]
)

analysis_models = analysis_models.sort_values(
    "General_Gap",
    ascending=False
)

display(analysis_models[
    ["Model", "General", "General_Gap"]
])

In [ ]:
import pandas as pd
import numpy as np

confusion_matrices = {
    "Logistic Regression": np.array([
        [289, 6, 5, 5, 24],
        [2, 94, 1, 0, 8],
        [3, 1, 126, 4, 24],
        [2, 2, 9, 257, 24],
        [27, 19, 24, 32, 257]
    ]),

    "BERT-base": np.array([
        [298, 3, 5, 0, 23],
        [2, 91, 2, 1, 9],
        [4, 1, 117, 4, 32],
        [2, 2, 6, 260, 24],
        [34, 17, 25, 34, 249]
    ]),

    "BioBERT": np.array([
        [301, 2, 5, 2, 19],
        [2, 92, 2, 2, 7],
        [2, 2, 127, 2, 25],
        [1, 3, 3, 268, 19],
        [28, 11, 28, 23, 269]
    ]),

    "BiomedBERT": np.array([
        [308, 2, 4, 2, 13],
        [3, 94, 2, 0, 6],
        [2, 0, 130, 1, 25],
        [0, 0, 2, 281, 11],
        [25, 12, 23, 31, 268]
    ]),

    "DeBERTa-v3": np.array([
        [312, 2, 3, 1, 11],
        [2, 92, 4, 1, 6],
        [4, 0, 125, 6, 23],
        [3, 2, 4, 264, 21],
        [41, 10, 24, 42, 242]
    ]),

    "Weighted BiLSTM": np.array([
        [276, 6, 6, 1, 40],
        [4, 77, 1, 0, 23],
        [6, 9, 80, 5, 58],
        [4, 5, 2, 250, 33],
        [32, 31, 23, 43, 230]
    ])
}

general_analysis = []

for model, cm in confusion_matrices.items():

    # Specific disease class incorrectly predicted as General
    specific_to_general = cm[:4, 4].sum()

    # True General incorrectly predicted as a specific disease class
    general_to_specific = cm[4, :4].sum()

    # Correct General predictions
    general_correct = cm[4, 4]

    # Total General examples
    general_total = cm[4, :].sum()

    # Recall for General
    general_recall = general_correct / general_total

    # All errors involving General in either direction
    total_general_related_errors = (
        specific_to_general + general_to_specific
    )

    general_analysis.append({
        "Model": model,
        "Specific → General": specific_to_general,
        "General → Specific": general_to_specific,
        "Total General-Related Errors": total_general_related_errors,
        "General Recall": general_recall
    })

general_analysis_df = pd.DataFrame(general_analysis)

general_analysis_df = general_analysis_df.sort_values(
    "Total General-Related Errors"
).reset_index(drop=True)

display(general_analysis_df)

In [ ]:
error_rows = []

for model, cm in confusion_matrices.items():

    total = cm.sum()
    correct = np.trace(cm)
    errors = total - correct

    general_related_errors = cm[:4, 4].sum() + cm[4, :4].sum()

    nervous_to_general = cm[2, 4]
    cardiovascular_to_general = cm[3, 4]
    neoplasms_to_general = cm[0, 4]
    digestive_to_general = cm[1, 4]

    general_to_neoplasms = cm[4, 0]
    general_to_digestive = cm[4, 1]
    general_to_nervous = cm[4, 2]
    general_to_cardiovascular = cm[4, 3]

    error_rows.append({
        "Model": model,
        "Total Errors": errors,
        "General-Related Errors": general_related_errors,
        "Nervous → General": nervous_to_general,
        "Cardiovascular → General": cardiovascular_to_general,
        "Neoplasms → General": neoplasms_to_general,
        "Digestive → General": digestive_to_general,
        "General → Neoplasms": general_to_neoplasms,
        "General → Digestive": general_to_digestive,
        "General → Nervous": general_to_nervous,
        "General → Cardiovascular": general_to_cardiovascular
    })

error_comparison_df = pd.DataFrame(error_rows)

error_comparison_df = error_comparison_df.sort_values(
    "Total Errors"
).reset_index(drop=True)

display(error_comparison_df)

In [ ]:
summary_cols = [
    "Model",
    "Total Errors",
    "General-Related Errors",
    "Nervous → General",
    "General → Cardiovascular"
]

display(
    error_comparison_df[summary_cols]
)

In [ ]:
error_comparison_df["General_Error_Share"] = (
    error_comparison_df["General-Related Errors"]
    / error_comparison_df["Total Errors"]
)

display(
    error_comparison_df[
        [
            "Model",
            "Total Errors",
            "General-Related Errors",
            "General_Error_Share"
        ]
    ]
)

In [ ]:
# Copy the existing splits
stage1_train_df = train_df.copy()
stage1_val_df = val_df.copy()

# Original label 5 = General
# New binary labels:
# 0 = Specific disease
# 1 = General pathological condition

stage1_train_df["binary_label"] = (
    stage1_train_df["condition_label"] == 5
).astype(int)

stage1_val_df["binary_label"] = (
    stage1_val_df["condition_label"] == 5
).astype(int)

print("TRAIN")
print(stage1_train_df["binary_label"].value_counts().sort_index())

print("\nVALIDATION")
print(stage1_val_df["binary_label"].value_counts().sort_index())

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BIOMEDBERT_CHECKPOINT = (
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

hier_tokenizer = AutoTokenizer.from_pretrained(
    BIOMEDBERT_CHECKPOINT
)

stage1_model = AutoModelForSequenceClassification.from_pretrained(
    BIOMEDBERT_CHECKPOINT,
    num_labels=2
).to(device)

print("Device:", device)
print("Stage 1 labels: Specific vs General")

In [ ]:
from torch.utils.data import Dataset, DataLoader

class BinaryMedicalDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length=512
    ):
        self.texts = (
            dataframe["medical_abstract"]
            .astype(str)
            .tolist()
        )

        self.labels = (
            dataframe["binary_label"]
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.long
                )
        }

In [ ]:
stage1_train_dataset = BinaryMedicalDataset(
    stage1_train_df,
    hier_tokenizer
)

stage1_val_dataset = BinaryMedicalDataset(
    stage1_val_df,
    hier_tokenizer
)

stage1_train_loader = DataLoader(
    stage1_train_dataset,
    batch_size=8,
    shuffle=True
)

stage1_val_loader = DataLoader(
    stage1_val_dataset,
    batch_size=8,
    shuffle=False
)

print("Train examples:", len(stage1_train_dataset))
print("Validation examples:", len(stage1_val_dataset))
print("Train batches:", len(stage1_train_loader))
print("Validation batches:", len(stage1_val_loader))

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

stage1_labels = stage1_train_df["binary_label"].values

stage1_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=stage1_labels
)

stage1_class_weights = torch.tensor(
    stage1_class_weights,
    dtype=torch.float32
).to(device)

print("Stage 1 class weights:", stage1_class_weights)

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from copy import deepcopy

STAGE1_EPOCHS = 3

stage1_optimizer = AdamW(
    stage1_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

total_steps = len(stage1_train_loader) * STAGE1_EPOCHS
warmup_steps = int(0.1 * total_steps)

stage1_scheduler = get_linear_schedule_with_warmup(
    stage1_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

stage1_criterion = torch.nn.CrossEntropyLoss(
    weight=stage1_class_weights
)

stage1_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

best_stage1_f1 = -1
best_stage1_state = None

print("Epochs:", STAGE1_EPOCHS)
print("Learning rate:", 2e-5)
print("Warmup steps:", warmup_steps)
print("Total steps:", total_steps)

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
train_df["binary_label"] = (train_df["condition_label"] == 5).astype(int)
val_df["binary_label"] = (val_df["condition_label"] == 5).astype(int)

print(train_df["binary_label"].value_counts())
print(val_df["binary_label"].value_counts())

In [ ]:
from torch.optim import AdamW

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
MAX_LEN = 512
BATCH_SIZE = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class Stage1Dataset(Dataset):
    def __init__(self, df):
        self.texts = df["medical_abstract"].astype(str).tolist()
        self.labels = df["binary_label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_stage1 = Stage1Dataset(train_df)
val_stage1 = Stage1Dataset(val_df)

train_loader = DataLoader(
    train_stage1,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_stage1,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

print("Train:", len(train_loader), "batches")
print("Validation:", len(val_loader), "batches")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

print("Ready")
print("Device:", device)
print("Total steps:", total_steps)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")

best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    model_stage1.train()
    train_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        with autocast("cuda"):
            outputs = model_stage1(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    model_stage1.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with autocast("cuda"):
                outputs = model_stage1(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    print(
        f"\nEpoch {epoch+1}: "
        f"Loss={train_loss/len(train_loader):.4f} | "
        f"Val Accuracy={val_acc:.4f} | "
        f"Val Macro F1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {
            k: v.cpu().clone()
            for k, v in model_stage1.state_dict().items()
        }

print("\nBest Stage 1 Macro F1:", round(best_f1, 4))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Restore best Stage 1 checkpoint
model_stage1.load_state_dict(best_state)
model_stage1.to(device)
model_stage1.eval()

stage1_preds = []
stage1_true = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with autocast("cuda"):
            outputs = model_stage1(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        preds = torch.argmax(outputs.logits, dim=1)

        stage1_preds.extend(preds.cpu().numpy())
        stage1_true.extend(labels.cpu().numpy())

print("Classification Report:")
print(
    classification_report(
        stage1_true,
        stage1_preds,
        target_names=["Specific", "General"],
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(stage1_true, stage1_preds))

In [ ]:
# Stage 2: classify only the 4 specific disease classes

# Keep only labels 1-4
train_stage2_df = train_df[train_df["condition_label"] != 5].copy()
val_stage2_df = val_df[val_df["condition_label"] != 5].copy()

# Map labels 1,2,3,4 -> 0,1,2,3
train_stage2_df["stage2_label"] = train_stage2_df["condition_label"] - 1
val_stage2_df["stage2_label"] = val_stage2_df["condition_label"] - 1

print("Train Stage 2 distribution:")
print(train_stage2_df["stage2_label"].value_counts().sort_index())

print("\nValidation Stage 2 distribution:")
print(val_stage2_df["stage2_label"].value_counts().sort_index())

print("\nTrain examples:", len(train_stage2_df))
print("Validation examples:", len(val_stage2_df))

In [ ]:
class Stage2Dataset(Dataset):
    def __init__(self, df):
        self.texts = df["medical_abstract"].astype(str).tolist()
        self.labels = df["stage2_label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_stage2_ds = Stage2Dataset(train_stage2_df)
val_stage2_ds = Stage2Dataset(val_stage2_df)

train_stage2_loader = DataLoader(
    train_stage2_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

val_stage2_loader = DataLoader(
    val_stage2_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

print("Stage 2 train batches:", len(train_stage2_loader))
print("Stage 2 validation batches:", len(val_stage2_loader))

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

stage2_classes = np.array([0, 1, 2, 3])

stage2_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=stage2_classes,
    y=train_stage2_df["stage2_label"].values
)

stage2_class_weights = torch.tensor(
    stage2_class_weights,
    dtype=torch.float
).to(device)

model_stage2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
).to(device)

optimizer_stage2 = AdamW(
    model_stage2.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

STAGE2_EPOCHS = 3

total_steps_stage2 = len(train_stage2_loader) * STAGE2_EPOCHS

scheduler_stage2 = get_linear_schedule_with_warmup(
    optimizer_stage2,
    num_warmup_steps=int(0.1 * total_steps_stage2),
    num_training_steps=total_steps_stage2
)

criterion_stage2 = torch.nn.CrossEntropyLoss(
    weight=stage2_class_weights
)

print("Stage 2 ready on:", device)
print("Class weights:", stage2_class_weights)
print("Total steps:", total_steps_stage2)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from torch.amp import autocast, GradScaler

scaler_stage2 = GradScaler("cuda")

best_stage2_f1 = 0
best_stage2_state = None

for epoch in range(STAGE2_EPOCHS):
    model_stage2.train()
    train_loss = 0

    progress_bar = tqdm(
        train_stage2_loader,
        desc=f"Stage 2 Epoch {epoch+1}/{STAGE2_EPOCHS}"
    )

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer_stage2.zero_grad()

        with autocast("cuda"):
            outputs = model_stage2(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion_stage2(
                outputs.logits,
                labels
            )

        scaler_stage2.scale(loss).backward()
        scaler_stage2.step(optimizer_stage2)
        scaler_stage2.update()
        scheduler_stage2.step()

        train_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    model_stage2.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_stage2_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with autocast("cuda"):
                outputs = model_stage2(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    print(
        f"\nEpoch {epoch+1}: "
        f"Loss={train_loss/len(train_stage2_loader):.4f} | "
        f"Val Accuracy={val_acc:.4f} | "
        f"Val Macro F1={val_f1:.4f}"
    )

    if val_f1 > best_stage2_f1:
        best_stage2_f1 = val_f1

        best_stage2_state = {
            k: v.cpu().clone()
            for k, v in model_stage2.state_dict().items()
        }

print(
    "\nBest Stage 2 Macro F1:",
    round(best_stage2_f1, 4)
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import numpy as np
import torch
from torch.amp import autocast

# Restore best checkpoints
model_stage1.load_state_dict(best_state)
model_stage1.to(device)
model_stage1.eval()

model_stage2.load_state_dict(best_stage2_state)
model_stage2.to(device)
model_stage2.eval()

hierarchical_preds = []

# We will process the full validation set using the original val_loader
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Stage 1: Specific vs General
        with autocast("cuda"):
            stage1_outputs = model_stage1(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        stage1_preds_batch = torch.argmax(stage1_outputs.logits, dim=1)

        # Stage 2: classify Specific cases into classes 1-4
        with autocast("cuda"):
            stage2_outputs = model_stage2(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        stage2_preds_batch = torch.argmax(stage2_outputs.logits, dim=1)

        for s1, s2 in zip(stage1_preds_batch.cpu().numpy(),
                          stage2_preds_batch.cpu().numpy()):

            if s1 == 1:
                # General
                hierarchical_preds.append(5)
            else:
                # Stage 2 labels 0-3 -> original labels 1-4
                hierarchical_preds.append(int(s2) + 1)

y_true = val_df["condition_label"].values

hier_acc = accuracy_score(y_true, hierarchical_preds)
hier_macro_f1 = f1_score(y_true, hierarchical_preds, average="macro")
hier_weighted_f1 = f1_score(y_true, hierarchical_preds, average="weighted")

print("Hierarchical Accuracy:", round(hier_acc, 4))
print("Hierarchical Macro F1:", round(hier_macro_f1, 4))
print("Hierarchical Weighted F1:", round(hier_weighted_f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        hierarchical_preds,
        target_names=[
            "Neoplasms",
            "Digestive",
            "Nervous",
            "Cardiovascular",
            "General"
        ],
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_true, hierarchical_preds))